<a href="https://colab.research.google.com/github/anushah-200/SATARK_AI/blob/main/notebooks/P3_diagnosis_correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!git clone https://github.com/anushah-200/SATARK_AI/tree/main

Cloning into 'main'...
fatal: repository 'https://github.com/anushah-200/SATARK_AI/tree/main/' not found


In [27]:
%cd /content
!git clone https://github.com/anushah-200/SATARK_AI.git
%cd /content/SATARK_AI
!git status

/content
Cloning into 'SATARK_AI'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 279 (delta 84), reused 88 (delta 52), pack-reused 133 (from 1)
Receiving objects: 100% (279/279), 27.15 MiB | 21.22 MiB/s, done.
Resolving deltas: 100% (119/119), done.
/content/SATARK_AI
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [6]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
import joblib


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
df = pd.read_csv("/content/drive/MyDrive/SATARK_AI/outputs/results/person2_day2_results.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


Shape: (43100, 26)

Columns:
['timestamp', 'station_id', 'station_name', 'temperature', 'humidity', 'pressure', 'wind_speed', 'temperature_deviation_24h', 'temperature_zscore_24h', 'temperature_change', 'absolute_temperature_change', 'temperature_std_6h', 'temperature_trend_6h', 'previous_6h_median', 'local_temperature_residual', 'previous_12h_median', 'local_residual_12h', 'repeat_length', 'rule_score', 'statistical_score', 'isolation_score', 'temporal_score', 'anomaly_score', 'fault_type', 'fault_severity', 'is_anomaly']


,timestamp,station_id,station_name,temperature,humidity,pressure,wind_speed,temperature_deviation_24h,temperature_zscore_24h,temperature_change,...,local_residual_12h,repeat_length,rule_score,statistical_score,isolation_score,temporal_score,anomaly_score,fault_type,fault_severity,is_anomaly
0,2025-01-01 00:00:00,42131,Hissar,5.4,97.0,1020.7,0.0,NaN,NaN,NaN,...,NaN,1,0,0,0,0,0,normal,none,0
1,2025-01-01 01:00:00,42131,Hissar,6.1,98.0,1019.5,5.0,NaN,NaN,0.7,...,NaN,2,0,0,0,0,0,normal,none,0
2,2025-01-01 02:00:00,42131,Hissar,6.1,94.0,1019.8,5.4,NaN,NaN,0.0,...,NaN,2,0,0,0,0,0,normal,none,0
3,2025-01-01 03:00:00,42131,Hissar,8.0,97.0,1021.5,0.0,2.133333,5.278631,1.9,...,NaN,1,0,1,1,0,2,normal,none,0
4,2025-01-01 04:00:00,42131,Hissar,7.6,96.0,1020.8,6.8,1.200000,1.074747,-0.4,...,NaN,1,0,0,1,0,1,normal,none,0


In [9]:
required_columns = [
    "timestamp",
    "station_id",
    "temperature"
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    print("Missing required columns:", missing)
else:
    print("All basic columns are present.")


All basic columns are present.


In [10]:
print(df.dtypes)


timestamp                       object
station_id                       int64
station_name                    object
temperature                    float64
humidity                       float64
pressure                       float64
wind_speed                     float64
temperature_deviation_24h      float64
temperature_zscore_24h         float64
temperature_change             float64
absolute_temperature_change    float64
temperature_std_6h             float64
temperature_trend_6h           float64
previous_6h_median             float64
local_temperature_residual     float64
previous_12h_median            float64
local_residual_12h             float64
repeat_length                    int64
rule_score                       int64
statistical_score                int64
isolation_score                  int64
temporal_score                   int64
anomaly_score                    int64
fault_type                      object
fault_severity                  object
is_anomaly               

In [11]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df = df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

print("Shape:", df.shape)
print("Stations:", df["station_id"].nunique())
print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())


Shape: (43100, 26)
Stations: 5
Time range: 2025-01-01 00:00:00 to 2026-01-01 00:00:00


In [12]:
NORMAL, SPIKE, MISSING, FROZEN, DRIFT, NOISE, SUSPICIOUS = (
    "NORMAL",
    "SPIKE",
    "MISSING",
    "FROZEN",
    "DRIFT",
    "NOISE",
    "SUSPICIOUS"
)

DETECTOR_COLS = [
    "rule_score",
    "statistical_score",
    "isolation_score",
    "temporal_score"
]


def _robust_threshold(series, k=5.0):
    """
    Robust threshold using Median + k * MAD.
    """
    series = series.dropna()

    if len(series) == 0:
        return 0

    med = series.median()
    mad = (series - med).abs().median()

    if mad > 0:
        return med + k * mad

    return med + k * series.std()


def diagnose_faults(
    df,
    spike_k=5.0,
    residual_k=5.0,
    frozen_min_repeat=4,
    frozen_std_pct=0.1,
    noise_flip_min=2,
    noise_std_k=1.0,
    drift_cumchange_k=3.0
):

    df = df.copy()

    df["diagnosed_fault"] = NORMAL
    df["diagnosis_confidence"] = 0.0

    for station, g in df.groupby("station_id"):

        idx = g.index

        # -----------------------------
        # Missing values
        # -----------------------------
        missing_mask = g["temperature"].isna()

        # -----------------------------
        # Thresholds
        # -----------------------------
        change_thr = _robust_threshold(
            g["absolute_temperature_change"],
            spike_k
        )

        residual_thr = _robust_threshold(
            g["local_temperature_residual"].abs(),
            residual_k
        )

        low_std_thr = (
            g["temperature_std_6h"]
            .dropna()
            .quantile(frozen_std_pct)
        )

        # -----------------------------
        # Recent temperature behaviour
        # -----------------------------
        change = g["temperature_change"].fillna(0)

        sign_flips = (
            (np.sign(change).diff().fillna(0) != 0)
            .astype(int)
            .rolling(6, min_periods=1)
            .sum()
        )

        cum_change_6h = (
            change
            .rolling(6, min_periods=1)
            .sum()
        )

        std_thr_noise = _robust_threshold(
            g["temperature_std_6h"],
            noise_std_k
        )

        cumchange_thr = _robust_threshold(
            cum_change_6h.abs(),
            drift_cumchange_k
        )

        # -----------------------------
        # SPIKE
        # -----------------------------
        spike_mask = (
            ~missing_mask
        ) & (
            (g["absolute_temperature_change"] > change_thr)
            |
            (g["local_temperature_residual"].abs() > residual_thr)
        )

        # -----------------------------
        # FROZEN
        # -----------------------------
        frozen_mask = (
            ~missing_mask
        ) & (
            ~spike_mask
        ) & (
            (g["repeat_length"] >= frozen_min_repeat)
            &
            (g["temperature_std_6h"] <= low_std_thr)
        )

        # -----------------------------
        # NOISE
        # -----------------------------
        noise_mask = (
            ~missing_mask
        ) & (
            ~spike_mask
        ) & (
            ~frozen_mask
        ) & (
            (sign_flips >= noise_flip_min)
            &
            (g["temperature_std_6h"] > std_thr_noise)
        )

        # -----------------------------
        # DRIFT
        # -----------------------------
        drift_mask = (
            ~missing_mask
        ) & (
            ~spike_mask
        ) & (
            ~frozen_mask
        ) & (
            ~noise_mask
        ) & (
            (cum_change_6h.abs() > cumchange_thr)
            &
            (sign_flips <= 2)
        )

        # -----------------------------
        # Detector agreement
        # -----------------------------
        detector_agree_count = (
            g[DETECTOR_COLS]
            .fillna(0)
            .gt(0)
            .sum(axis=1)
        )

        detector_fired = detector_agree_count >= 1

        # P2 detector confirmation
        spike_mask = spike_mask & detector_fired
        frozen_mask = frozen_mask & detector_fired
        drift_mask = drift_mask & detector_fired

        # -----------------------------
        # Suspicious
        # -----------------------------
        confirmed = (
            missing_mask
            | spike_mask
            | frozen_mask
            | noise_mask
            | drift_mask
        )

        suspicious_mask = (
            ~confirmed
        ) & detector_fired

        # -----------------------------
        # Assign labels
        # -----------------------------
        labels = pd.Series(
            NORMAL,
            index=idx
        )

        labels[suspicious_mask] = SUSPICIOUS
        labels[drift_mask] = DRIFT
        labels[noise_mask] = NOISE
        labels[frozen_mask] = FROZEN
        labels[spike_mask] = SPIKE
        labels[missing_mask] = MISSING

        df.loc[idx, "diagnosed_fault"] = labels

        # -----------------------------
        # Confidence
        # -----------------------------
        conf = pd.Series(
            0.0,
            index=idx
        )

        conf[missing_mask] = 1.0

        if change_thr > 0:

            conf[spike_mask] = (
                g.loc[
                    spike_mask,
                    "absolute_temperature_change"
                ]
                / change_thr
            ).clip(upper=2) / 2

        if low_std_thr > 0:

            conf[frozen_mask] = (
                1 -
                (
                    g.loc[
                        frozen_mask,
                        "temperature_std_6h"
                    ]
                    / low_std_thr
                ).clip(upper=1)
            )

        if std_thr_noise > 0:

            conf[noise_mask] = (
                g.loc[
                    noise_mask,
                    "temperature_std_6h"
                ]
                / std_thr_noise
            ).clip(upper=2) / 2

        if cumchange_thr > 0:

            conf[drift_mask] = (
                cum_change_6h[drift_mask].abs()
                / cumchange_thr
            ).clip(upper=2) / 2

        conf[suspicious_mask] = 0.4

        df.loc[
            idx,
            "diagnosis_confidence"
        ] = conf.round(2)

    return df



In [13]:
df = diagnose_faults(df)

print("Diagnosis completed.")
print()
print(df["diagnosed_fault"].value_counts())


Diagnosis completed.

diagnosed_fault
NORMAL        32460
SUSPICIOUS     4891
NOISE          3637
SPIKE          1045
DRIFT           753
FROZEN          313
MISSING           1
Name: count, dtype: int64


In [14]:
diagnosis_summary = (
    df["diagnosed_fault"]
    .value_counts()
    .rename_axis("diagnosis")
    .reset_index(name="count")
)

diagnosis_summary


,diagnosis,count
0,NORMAL,32460
1,SUSPICIOUS,4891
2,NOISE,3637
3,SPIKE,1045
4,DRIFT,753
5,FROZEN,313
6,MISSING,1


In [15]:
print(
    "Flagged rows:",
    (df["diagnosed_fault"] != "NORMAL").sum()
)

print(
    "Total rows:",
    len(df)
)


Flagged rows: 10640
Total rows: 43100


In [16]:
def correct_temperature(df, min_confidence=0.5):

    df = df.sort_values(
        ["station_id", "timestamp"]
    ).reset_index(drop=True)

    # Preserve the observed temperature
    df["temperature_original"] = df["temperature"]

    # Start corrected values as the original values
    df["temperature_corrected"] = df["temperature"]

    # Track whether correction happened
    df["correction_applied"] = 0

    for station, g in df.groupby("station_id"):

        idx = g.index

        temp = g["temperature"].copy()

        # --------------------------------------------------
        # MISSING
        # --------------------------------------------------
        missing_mask = (
            g["diagnosed_fault"] == MISSING
        )

        if missing_mask.any():

            interpolated = (
                temp
                .interpolate(
                    method="linear",
                    limit_direction="both"
                )
            )

            temp[missing_mask] = (
                interpolated[missing_mask]
            )

        # --------------------------------------------------
        # SPIKE
        # --------------------------------------------------
        spike_mask = (
            g["diagnosed_fault"] == SPIKE
        )

        if spike_mask.any():

            baseline = (
                g["previous_6h_median"]
                .fillna(
                    g["previous_12h_median"]
                )
            )

            temp[spike_mask] = (
                baseline[spike_mask]
            )

        # --------------------------------------------------
        # FROZEN
        # --------------------------------------------------
        frozen_mask = (
            g["diagnosed_fault"] == FROZEN
        )

        if frozen_mask.any():

            temp_for_interp = temp.copy()

            temp_for_interp[frozen_mask] = np.nan

            interpolated = (
                temp_for_interp
                .interpolate(
                    method="linear",
                    limit_direction="both"
                )
            )

            temp[frozen_mask] = (
                interpolated[frozen_mask]
            )

        # --------------------------------------------------
        # NOISE
        # --------------------------------------------------
        noise_mask = (
            g["diagnosed_fault"] == NOISE
        )

        if noise_mask.any():

            baseline = (
                g["previous_6h_median"]
            )

            temp[noise_mask] = (
                baseline[noise_mask]
            )

        # --------------------------------------------------
        # DRIFT
        # --------------------------------------------------
        drift_mask = (
            g["diagnosed_fault"] == DRIFT
        )

        if drift_mask.any():

            baseline = (
                g["previous_12h_median"]
            )

            temp[drift_mask] = (
                baseline[drift_mask]
            )

        # --------------------------------------------------
        # Apply only sufficiently confident corrections
        # --------------------------------------------------
        actionable = g["diagnosed_fault"].isin(
            [
                MISSING,
                SPIKE,
                FROZEN,
                NOISE,
                DRIFT
            ]
        )

        confident = (
            g["diagnosis_confidence"]
            >= min_confidence
        )

        apply_mask = (
            actionable
            &
            (
                confident
                |
                (g["diagnosed_fault"] == MISSING)
            )
        )

        final_temp = g["temperature"].copy()

        final_temp[apply_mask] = (
            temp[apply_mask]
        )

        df.loc[
            idx,
            "temperature_corrected"
        ] = final_temp.values

        df.loc[
            idx[apply_mask],
            "correction_applied"
        ] = 1

    return df


In [17]:
df = correct_temperature(df)

print("Correction completed.")

print(
    "Rows corrected:",
    df["correction_applied"].sum()
)


Correction completed.
Rows corrected: 5250


In [18]:
df[
    [
        "timestamp",
        "station_id",
        "temperature",
        "diagnosed_fault",
        "diagnosis_confidence",
        "temperature_corrected",
        "correction_applied"
    ]
].head(20)


,timestamp,station_id,temperature,diagnosed_fault,diagnosis_confidence,temperature_corrected,correction_applied
0,2025-01-01 00:00:00,42131,5.4,NORMAL,0.00,5.40,0
1,2025-01-01 01:00:00,42131,6.1,NORMAL,0.00,6.10,0
2,2025-01-01 02:00:00,42131,6.1,NORMAL,0.00,6.10,0
3,2025-01-01 03:00:00,42131,8.0,SUSPICIOUS,0.40,8.00,0
4,2025-01-01 04:00:00,42131,7.6,SUSPICIOUS,0.40,7.60,0
5,2025-01-01 05:00:00,42131,9.8,SUSPICIOUS,0.40,9.80,0
6,2025-01-01 06:00:00,42131,10.0,SUSPICIOUS,0.40,10.00,0
7,2025-01-01 07:00:00,42131,14.9,SPIKE,0.74,7.80,1
8,2025-01-01 08:00:00,42131,16.3,NOISE,0.81,8.90,1
9,2025-01-01 09:00:00,42131,11.8,NORMAL,0.00,11.80,0


In [19]:
print(
    df.groupby("diagnosed_fault")["correction_applied"]
    .agg(["count", "sum"])
)


                 count   sum
diagnosed_fault             
DRIFT              753   753
FROZEN             313   191
MISSING              1     1
NOISE             3637  3637
NORMAL           32460     0
SPIKE             1045   668
SUSPICIOUS        4891     0


In [20]:
df = correct_temperature(df)


In [21]:
df[[
    "timestamp",
    "station_id",
    "temperature",
    "diagnosed_fault",
    "diagnosis_confidence",
    "temperature_corrected",
    "correction_applied"
]].head(20)



,timestamp,station_id,temperature,diagnosed_fault,diagnosis_confidence,temperature_corrected,correction_applied
0,2025-01-01 00:00:00,42131,5.4,NORMAL,0.00,5.40,0
1,2025-01-01 01:00:00,42131,6.1,NORMAL,0.00,6.10,0
2,2025-01-01 02:00:00,42131,6.1,NORMAL,0.00,6.10,0
3,2025-01-01 03:00:00,42131,8.0,SUSPICIOUS,0.40,8.00,0
4,2025-01-01 04:00:00,42131,7.6,SUSPICIOUS,0.40,7.60,0
5,2025-01-01 05:00:00,42131,9.8,SUSPICIOUS,0.40,9.80,0
6,2025-01-01 06:00:00,42131,10.0,SUSPICIOUS,0.40,10.00,0
7,2025-01-01 07:00:00,42131,14.9,SPIKE,0.74,7.80,1
8,2025-01-01 08:00:00,42131,16.3,NOISE,0.81,8.90,1
9,2025-01-01 09:00:00,42131,11.8,NORMAL,0.00,11.80,0


In [22]:
df["correction_applied"].value_counts()


,count
correction_applied,
0,37850
1,5250


In [23]:
pd.crosstab(
    df["diagnosed_fault"],
    df["correction_applied"]
)


correction_applied,0,1
diagnosed_fault,,
DRIFT,0,753
FROZEN,122,191
MISSING,0,1
NOISE,0,3637
NORMAL,32460,0
SPIKE,377,668
SUSPICIOUS,4891,0


In [24]:
import os

output_dir = "/content/drive/MyDrive/SATARK_AI/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "person3_diagnosis_correction_handover.csv"
)

df.to_csv(output_path, index=False)

print("Saved successfully!")
print("Location:", output_path)
print("Shape:", df.shape)


Saved successfully!
Location: /content/drive/MyDrive/SATARK_AI/outputs/person3_diagnosis_correction_handover.csv
Shape: (43100, 31)


In [25]:
df[[
    "timestamp",
    "station_id",
    "temperature",
    "diagnosed_fault",
    "diagnosis_confidence",
    "temperature_corrected",
    "correction_applied"
]].head()


,timestamp,station_id,temperature,diagnosed_fault,diagnosis_confidence,temperature_corrected,correction_applied
0,2025-01-01 00:00:00,42131,5.4,NORMAL,0.0,5.4,0
1,2025-01-01 01:00:00,42131,6.1,NORMAL,0.0,6.1,0
2,2025-01-01 02:00:00,42131,6.1,NORMAL,0.0,6.1,0
3,2025-01-01 03:00:00,42131,8.0,SUSPICIOUS,0.4,8.0,0
4,2025-01-01 04:00:00,42131,7.6,SUSPICIOUS,0.4,7.6,0


In [28]:
%cd /content/SATARK_AI
!git checkout -- results/person3_diagnosis_correction.csv results/person3_diagnosis_correction_eval.csv
!git status

/content/SATARK_AI
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [29]:
!pwd

/content/SATARK_AI


In [30]:
import pandas as pd

p3 = pd.read_csv("results/person3_diagnosis_correction.csv")
meta = pd.read_csv("outputs/results/anomaly_detection_results.csv")

print("=== person3_diagnosis_correction.csv ===")
print("station_id dtype:", p3["station_id"].dtype)
print("station_id sample:", p3["station_id"].unique()[:5])
print("timestamp dtype:", p3["timestamp"].dtype)
print("timestamp sample:", p3["timestamp"].head(3).tolist())
print()

print("=== anomaly_detection_results.csv ===")
print("station_id dtype:", meta["station_id"].dtype)
print("station_id sample:", meta["station_id"].unique()[:5])
print("timestamp dtype:", meta["timestamp"].dtype)
print("timestamp sample:", meta["timestamp"].head(3).tolist())
print()

# Simulate her actual merge
p3["timestamp"] = pd.to_datetime(p3["timestamp"], errors="coerce")
meta["timestamp"] = pd.to_datetime(meta["timestamp"], errors="coerce")

merged = p3.merge(
    meta[["timestamp", "station_id", "station_name", "latitude", "longitude"]].drop_duplicates(["timestamp","station_id"]),
    on=["timestamp", "station_id"],
    how="left"
)

print("Rows in p3:", len(p3))
print("Rows with lat/lon successfully matched after merge:", merged["latitude"].notna().sum())

=== person3_diagnosis_correction.csv ===
station_id dtype: int64
station_id sample: [42131 42139 42176 42181 42182]
timestamp dtype: object
timestamp sample: ['2025-01-01 00:00:00', '2025-01-01 01:00:00', '2025-01-01 02:00:00']

=== anomaly_detection_results.csv ===
station_id dtype: int64
station_id sample: [42131 42139 42176 42181 42182]
timestamp dtype: object
timestamp sample: ['2025-01-01 00:00:00', '2025-01-01 01:00:00', '2025-01-01 02:00:00']

Rows in p3: 43100
Rows with lat/lon successfully matched after merge: 43100


In [31]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 96.4 MB/s eta 0:00:00


In [32]:
!wget -q -O - ipv4.icanhazip.com

136.112.163.190


In [33]:
!streamlit run /content/SATARK_AI/dashboard/app.py &>/content/logs.txt &

⠙⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋^C


In [35]:
!cat /content/logs.txt



2026-09-12 07:19:33.148 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.112.163.190:8501

  Stopping...


In [37]:
!streamlit run /content/SATARK_AI/dashboard/app.py &>/content/logs.txt &

In [38]:
!sleep 3 && cat /content/logs.txt



2026-09-12 07:25:41.494 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.112.163.190:8501



In [39]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://true-wolves-swim.loca.lt
^C


In [40]:
!cat /content/logs.txt



2026-09-12 07:25:41.494 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.112.163.190:8501

  Stopping...


In [41]:
!ps aux | grep streamlit

root        8744  0.0  0.0   7340  3432 ?        S    07:35   0:00 /bin/bash -c ps aux | grep streamlit
root        8746  0.0  0.0   6544  2284 ?        S    07:35   0:00 grep streamlit


In [42]:
!nohup streamlit run /content/SATARK_AI/dashboard/app.py --server.address=0.0.0.0 &> /content/logs.txt &
!sleep 5
!cat /content/logs.txt



2026-09-12 07:35:52.257 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.112.163.190:8501



In [43]:
!ps aux | grep streamlit

root        8860  8.3  0.5 525460 71876 ?        Sl   07:35   0:01 /usr/bin/python3 /usr/local/bin/streamlit run /content/SATARK_AI/dashboard/app.py --server.address=0.0.0.0
root        8952  0.0  0.0   7340  3552 ?        S    07:36   0:00 /bin/bash -c ps aux | grep streamlit
root        8954  0.0  0.0   6544  2320 ?        S    07:36   0:00 grep streamlit


In [44]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦your url is: https://twenty-towns-admire.loca.lt
^C


ERROR:pyngrok.process.ngrok:t=2026-09-12T07:39:39+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.